# 🦙 LEVEL 1 — LlamaIndex Fundamentals

### Level: 🟢 Beginner

---

## Prerequisites
- Basic Python knowledge
- Basic understanding of what an LLM (Large Language Model) is
- An OpenAI or Google Gemini API key

## What You Will Learn
- What LlamaIndex is and WHY it was created
- The complete LlamaIndex architecture
- Every core abstraction: Document, Node, Index, Retriever, Query Engine, Chat Engine, Response Synthesizer, Tools, Agents, Workflows
- How all these pieces fit together
- The complete LlamaIndex execution lifecycle

## Why This Level Matters
Everything in LlamaIndex builds on these fundamentals. Understanding the WHY behind each abstraction is the difference between a developer who copy-pastes code and one who designs production systems.

---

## Topic 1 — What is LlamaIndex?

### 1. What is it?
LlamaIndex is a **data framework for building LLM-powered applications that need to work with your own data**.

> LlamaIndex is the bridge between your private data (PDFs, databases, APIs, files) and LLMs (GPT-4, Gemini, Claude).

### 2. Why does LlamaIndex exist?

**The core problem:** LLMs only know what they were trained on. They don't know your company's internal documents, your private database, or any data after their training cutoff. You CANNOT just dump 10,000 pages into an LLM — there is a token limit, cost, and latency problem.

**The solution:** Store data in a searchable index → retrieve ONLY relevant pieces → give LLM only that context → get grounded answer. LlamaIndex handles all of this.

### 3. Mental Model
LlamaIndex is a **very smart librarian**:
- Library = your data
- Card catalog = the index
- Librarian = the retriever
- Scholar = the LLM reading the relevant pages

### 4. The Big Picture Architecture

```
                     LlamaIndex
                         |
           +-------------+-------------+
           |             |             |
        Data          Indexing       Agents
           |             |             |
       Documents      Retrieval      Tools
           |             |             |
         Nodes        Query Engine   Workflows
           |             |
           +------+------+
                  |
                 LLM
                  |
                Answer
```

### 5. The Complete Execution Lifecycle

```
INGESTION PHASE (once / periodically)
=====================================
Raw Data → Loader → Document → Node Parser
→ Nodes → Embedding Model → Vector Store / Index

QUERY PHASE (per user query)
=============================
User Question → Query Embedding → Retriever
→ Relevant Nodes → Postprocessor
→ Response Synthesizer → LLM → Answer
```

## Topic 2 — The 10 Core Abstractions

| Abstraction | One-Line Purpose |
|-------------|------------------|
| **Document** | Container for raw text + metadata from a source |
| **Node** | A chunk of a Document — the atomic unit of retrieval |
| **Index** | Data structure that organizes Nodes for fast retrieval |
| **Retriever** | Fetches the most relevant Nodes for a query |
| **Response Synthesizer** | Combines retrieved Nodes + query → LLM prompt → answer |
| **Query Engine** | End-to-end: takes a question, returns an answer |
| **Chat Engine** | Like Query Engine but with memory (multi-turn) |
| **Tool** | A callable function an Agent can use |
| **Agent** | An LLM that dynamically decides which Tools to call |
| **Workflow** | An explicit event-driven pipeline of steps you define |

---

## Topic 3 — Document

### What is it?
A `Document` is LlamaIndex's container for raw data. Every loader outputs Documents.

### Architecture
```
PDF File → PDFReader → Document
                          ├── text: "Full text content..."
                          ├── metadata: {file_name, page, author, ...}
                          ├── doc_id: "unique-id"
                          └── embedding: None  (set later)
```

### Common Mistakes
- ❌ Ignoring metadata — crucial for filtering
- ❌ Not setting doc_id — makes updates impossible
- ❌ Treating Document as the retrieval unit — Nodes are retrieved, not Documents

In [ ]:
from llama_index.core import Document

# Minimal Document
doc1 = Document(text="LlamaIndex is a data framework for LLM applications.")
print(f"Text    : {doc1.text}")
print(f"Doc ID  : {doc1.doc_id}")
print(f"Metadata: {doc1.metadata}")
print()

In [ ]:
# Document with metadata (real-world pattern)
doc2 = Document(
    text="Q4 2024 Revenue: $5.2M. Growth: 23% YoY. Top Product: Enterprise Plan.",
    metadata={
        "source": "finance_report_q4_2024.pdf",
        "author": "Finance Team",
        "quarter": "Q4",
        "year": "2024",
        "department": "Finance"
    },
    doc_id="finance-q4-2024-001"
)

print(f"Doc ID  : {doc2.doc_id}")
print(f"Metadata: {doc2.metadata}")

# Exclude sensitive fields from embedding / LLM context
doc2.excluded_embed_metadata_keys = ["author"]
doc2.excluded_llm_metadata_keys = ["author"]
print(f"Excluded from embed: {doc2.excluded_embed_metadata_keys}")

In [ ]:
# Loading Documents from directory
from llama_index.core import SimpleDirectoryReader

docs = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(docs)} document(s)")
for d in docs:
    print(f"  ID: {d.doc_id} | Metadata: {d.metadata}")

## Topic 4 — Node

### What is it?
A `Node` is a chunk of a Document — the atomic unit of retrieval. The Retriever retrieves Nodes, not Documents.

### Why does it exist?
A Document can be 50 pages. An LLM has a token limit. You need to identify which 2-3 paragraphs are relevant. Split Documents into Nodes → embed each → retrieve most relevant.

### Architecture
```
Document (50 pages)
    |
    v
Node Parser
    ├── Node 1: text, node_id, metadata, embedding, relationships
    ├── Node 2: text, node_id, metadata, embedding, relationships
    └── Node 3: text, node_id, metadata, embedding, relationships
```

### Document vs Node

| Property | Document | Node |
|----------|----------|------|
| Purpose | Container for raw data | Atomic retrieval unit |
| Size | Entire file | Small chunk |
| Has embedding? | No | Yes |
| Has relationships? | No | Yes (parent, prev, next) |
| Created by | Loaders | Node Parsers |
| Retrieved? | Never | Always |

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

doc = Document(
    text="""LlamaIndex is a powerful data framework.
    It was built to connect LLMs with private data.
    Core components: Documents, Nodes, Indexes, and Retrievers.
    Documents are the raw input. Nodes are atomic retrieval units.
    Indexes organize Nodes for fast lookup. Retrievers find relevant Nodes.""",
    metadata={"source": "intro_guide.txt"}
)

parser = SentenceSplitter(chunk_size=100, chunk_overlap=20)
nodes = parser.get_nodes_from_documents([doc])

print(f"Document length: {len(doc.text)} chars")
print(f"Nodes created  : {len(nodes)}")
print()

for i, node in enumerate(nodes):
    print(f"Node {i+1}:")
    print(f"  ID      : {node.node_id}")
    print(f"  Text    : {node.text.strip()}")
    print(f"  Metadata: {node.metadata}")
    print(f"  Embedding: {node.embedding}  (None until indexed)")
    print()

In [ ]:
# Node relationships — nodes know where they came from
print("Node Relationships:")
for i, node in enumerate(nodes):
    print(f"  Node {i+1}:")
    for rel_type, rel_info in node.relationships.items():
        print(f"    {rel_type.name}: {rel_info.node_id[:8]}...")

## Topic 5 — Index

### What is it?
An `Index` organizes Nodes (with their embeddings) to enable fast, relevant retrieval.

### Architecture
```
Nodes → Embedding Model → Vectors
                              |
                         Vector Store
                              |
                      VectorStoreIndex  ← LlamaIndex abstraction
                              |
                         Retriever
```

### Index Types
| Index | Best For |
|-------|----------|
| `VectorStoreIndex` | Semantic search (most common) |
| `SummaryIndex` | Summarizing all content |
| `KeywordTableIndex` | Exact keyword matching |
| `SQLStructStoreIndex` | Structured SQL data |

### Common Mistakes
- ❌ Rebuilding index on every restart (expensive!) — persist it
- ❌ Not configuring embedding model explicitly

In [ ]:
import os
# Set your API key:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

from llama_index.core import VectorStoreIndex, Document

documents = [
    Document(text="LlamaIndex helps you build RAG applications with your own data.",
             metadata={"topic": "LlamaIndex"}),
    Document(text="Embeddings are numerical vector representations of text.",
             metadata={"topic": "Embeddings"}),
    Document(text="Vector databases store embeddings and allow similarity search.",
             metadata={"topic": "Vector Stores"}),
    Document(text="RAG stands for Retrieval-Augmented Generation.",
             metadata={"topic": "RAG"}),
]

# Automatically: parses → embeds → stores
index = VectorStoreIndex.from_documents(documents)

print(f"Index type : {type(index).__name__}")
print(f"Nodes count: {len(index.docstore.docs)}")

## Topic 6 — Retriever

### What is it?
Given a user query, the `Retriever` finds and returns the most relevant Nodes from an Index.

### Architecture
```
Query: "What is RAG?" → Embedding → Query Vector
    |
    v
Vector Store similarity search:
  Node 1: similarity = 0.95  ← Top!
  Node 2: similarity = 0.82
  Node 3: similarity = 0.41  (not relevant)
    |
    v
Returns top-K Nodes
```

### Retriever vs Query Engine
| | Retriever | Query Engine |
|--|-----------|---------------|
| Output | Raw Nodes | Final Answer (string) |
| Uses LLM? | No | Yes |
| Use when | Debugging / building | User-facing answers |

In [ ]:
# Build retriever from index
retriever = index.as_retriever(similarity_top_k=2)

query = "What is RAG?"
retrieved_nodes = retriever.retrieve(query)

print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved_nodes)} nodes:")
print()

for i, node_with_score in enumerate(retrieved_nodes):
    print(f"Result {i+1}:")
    print(f"  Score   : {node_with_score.score:.4f}")
    print(f"  Text    : {node_with_score.node.text}")
    print(f"  Metadata: {node_with_score.node.metadata}")
    print()

## Topic 7 — Query Engine

### What is it?
End-to-end Q&A: give it a question, get a natural language answer. Internally: Retriever + Response Synthesizer.

### Architecture
```
User Query
    |
    v
Query Engine
    ├── Retriever → Retrieved Nodes
    └── Response Synthesizer → [Query + Nodes] → LLM → Answer
```

### Query Engine vs Chat Engine
| | Query Engine | Chat Engine |
|--|--------------|-------------|
| Memory | No (stateless) | Yes |
| Multi-turn | No | Yes |
| Interface | `.query()` | `.chat()` |

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=2)

response = query_engine.query("What is RAG?")

print(f"Answer: {response}")
print()
print("Source nodes used:")
for src in response.source_nodes:
    print(f"  [{src.score:.3f}] {src.node.text}")

## Topic 8 — Chat Engine

### What is it?
A conversational interface over your index with memory — remembers previous messages for context-aware responses.

### Architecture
```
Turn 1: "What is LlamaIndex?" → retrieves + answers → stores in memory
Turn 2: "What are its main components?" → condenses using history
        → becomes "What are LlamaIndex's main components?"
        → retrieves + answers
```

### Chat Modes
| Mode | Description |
|------|-------------|
| `condense_question` | Reformulates follow-ups using history |
| `context` | Always retrieves fresh context |
| `condense_plus_context` | Condense first, then retrieve |
| `simple` | Pure LLM chat, no retrieval |

In [ ]:
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    verbose=True
)

print("=== Turn 1 ===")
r1 = chat_engine.chat("What is LlamaIndex?")
print(f"Assistant: {r1}")
print()

print("=== Turn 2 (follow-up) ===")
r2 = chat_engine.chat("What are its main use cases?")
print(f"Assistant: {r2}")

chat_engine.reset()  # Reset conversation
print("\nChat reset.")

## Topic 9 — Response Synthesizer

### What is it?
Takes retrieved Nodes + the query → constructs LLM prompt → processes response.

### Response Modes
```
compact        → Combine all nodes → Single LLM call (fastest)
refine         → Node 1 → Answer 1 → Answer 1 + Node 2 → Refined Answer → ...
tree_summarize → Build summary tree (best for large node sets)
no_text        → Return only source nodes (debugging)
```

| Mode | Use When |
|------|----------|
| `compact` | Standard Q&A, speed matters |
| `refine` | Accuracy critical |
| `tree_summarize` | Summarizing large docs |
| `no_text` | Debugging retrieval |

In [ ]:
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import ResponseMode

retriever = index.as_retriever(similarity_top_k=3)
query = "What are embeddings and why are they used?"
nodes = retriever.retrieve(query)

# compact mode
synth = get_response_synthesizer(response_mode=ResponseMode.COMPACT)
response = synth.synthesize(query, nodes=nodes)
print(f"COMPACT: {response}")
print()

# refine mode
synth_refine = get_response_synthesizer(response_mode=ResponseMode.REFINE)
response_refined = synth_refine.synthesize(query, nodes=nodes)
print(f"REFINE: {response_refined}")

## Topic 10 — Tools

### What is it?
A `Tool` is a callable function an Agent can use. Tools are how agents interact with query engines, APIs, databases, etc.

### Tool Types
| Tool Type | Description |
|-----------|-------------|
| `FunctionTool` | Wraps any Python function |
| `QueryEngineTool` | Wraps a QueryEngine |
| `RetrieverTool` | Wraps a Retriever |

> **Critical:** The LLM selects tools based on their NAME and DESCRIPTION. Write clear descriptions!

In [ ]:
from llama_index.core.tools import FunctionTool, QueryEngineTool

# FunctionTool wraps any Python function
def get_word_count(text: str) -> int:
    """Returns the number of words in the given text."""
    return len(text.split())

word_count_tool = FunctionTool.from_defaults(
    fn=get_word_count,
    name="word_counter",
    description="Counts words in a text. Use when the user asks about word counts."
)

result = word_count_tool("LlamaIndex is a powerful framework for building LLM apps")
print(f"Word count: {result}")
print(f"Tool name : {word_count_tool.metadata.name}")
print()

# QueryEngineTool wraps a query engine as a tool
qe_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="llamaindex_kb",
    description="Answers questions about LlamaIndex: Documents, Nodes, Indexes, RAG."
)
print(f"QE Tool: {qe_tool.metadata.name}")

## Topic 11 — Agents

### What is it?
An Agent is an LLM that reasons, plans, and dynamically calls Tools until it can answer. Unlike Query Engines (fixed pipeline), Agents decide at runtime what to do.

### The Agent Loop
```
User Question → Agent
    → THINK: Which tool do I need?
    → CALL Tool A → Observe result
    → THINK: Do I have enough? Need more?
    → CALL Tool B (if needed) → Observe
    → Generate Final Answer
```

### Agent vs Query Engine vs Workflow
| | Query Engine | Agent | Workflow |
|--|--------------|-------|----------|
| Control flow | Fixed | LLM decides | You define |
| Steps | 1 | Dynamic | Explicit |
| Predictable? | Yes | No | Yes |
| Use when | Simple Q&A | Complex multi-step | Production |

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import FunctionTool

def add(a: float, b: float) -> float:
    """Adds two numbers and returns the result."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiplies two numbers and returns the result."""
    return a * b

agent = ReActAgent.from_tools(
    tools=[
        FunctionTool.from_defaults(fn=add, name="add"),
        FunctionTool.from_defaults(fn=multiply, name="multiply"),
        qe_tool,
    ],
    verbose=True,
    max_iterations=10
)

response = agent.chat("What is (15 + 7) multiplied by 3?")
print(f"\nFinal Answer: {response}")

## Topic 12 — Workflows

### What is it?
An explicit, event-driven pipeline where YOU define the steps and routing. Unlike Agents (LLM decides), Workflows are deterministic.

### Core Concepts
- **Event** — message that triggers a Step
- **Step** — `@step`-decorated function that processes an event and emits another
- **StartEvent** — triggers the workflow
- **StopEvent** — ends the workflow
- **Context** — shared state all steps can read/write

### Architecture
```
StartEvent → Step:retrieve → RetrievedEvent
                                 |
                            Step:rerank → RerankedEvent
                                              |
                                         Step:synthesize → StopEvent
```

### Agent vs Workflow
| | Agent | Workflow |
|--|-------|----------|
| Who decides steps? | LLM | Developer |
| Deterministic? | No | Yes |
| Debug-friendly? | Hard | Easy |
| Best for | Exploration | Production |

In [ ]:
from llama_index.core.workflow import (
    Workflow, step, Event, StartEvent, StopEvent, Context
)

# Custom event: message between steps
class QueryProcessedEvent(Event):
    query: str
    context: str

class SimpleRAGWorkflow(Workflow):

    @step
    async def retrieve(self, ctx: Context, ev: StartEvent) -> QueryProcessedEvent:
        query = ev.get("query", "")
        print(f"[Step 1: retrieve] query='{query}'")
        context = f"Retrieved context for: {query}"  # would call real retriever
        return QueryProcessedEvent(query=query, context=context)

    @step
    async def synthesize(self, ctx: Context, ev: QueryProcessedEvent) -> StopEvent:
        print(f"[Step 2: synthesize] generating answer...")
        answer = f"Answer based on: '{ev.context}'"
        return StopEvent(result=answer)

import asyncio

async def run():
    wf = SimpleRAGWorkflow(timeout=30)
    result = await wf.run(query="What is LlamaIndex?")
    print(f"\nResult: {result}")

await run()

## Topic 13 — Settings (Global Configuration)

In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# Configure LLM
import os
from dotenv import load_dotenv
load_dotenv()

Settings.llm = GoogleGenAI(
    model="gemini-2.0-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

Settings.embed_model = GoogleGenAIEmbedding(
    model_name="models/text-embedding-004",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

# Chunking defaults
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM         : {Settings.llm}")
print(f"Embed Model : {Settings.embed_model}")
print(f"Chunk Size  : {Settings.chunk_size}")
print(f"Chunk Overlap: {Settings.chunk_overlap}")


## Topic 14 — Complete Level 1 Pipeline

In [ ]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.core.node_parser import SentenceSplitter

# STEP 1: Documents
docs = [
    Document(text="LlamaIndex is a data framework for building LLM-powered apps.",
             metadata={"topic": "overview"}),
    Document(text="A Document wraps raw text data with metadata.",
             metadata={"topic": "documents"}),
    Document(text="Nodes are atomic retrieval units — chunks of a Document.",
             metadata={"topic": "nodes"}),
    Document(text="VectorStoreIndex stores embeddings for semantic search.",
             metadata={"topic": "indexing"}),
    Document(text="RAG = Retrieval-Augmented Generation. LlamaIndex makes it easy.",
             metadata={"topic": "RAG"}),
]
print(f"Step 1 ✅ {len(docs)} Documents")

# STEP 2: Nodes
parser = SentenceSplitter(chunk_size=128, chunk_overlap=20)
nodes = parser.get_nodes_from_documents(docs)
print(f"Step 2 ✅ {len(nodes)} Nodes")

# STEP 3: Index
index = VectorStoreIndex(nodes)
print(f"Step 3 ✅ Index built")

# STEP 4: Retrieve
retriever = index.as_retriever(similarity_top_k=2)
found = retriever.retrieve("What are Nodes?")
print(f"Step 4 ✅ Retriever found {len(found)} nodes")
for r in found:
    print(f"   [{r.score:.3f}] {r.node.text[:60].strip()}")

# STEP 5: Query
qe = index.as_query_engine(similarity_top_k=2)
ans = qe.query("What is the purpose of Nodes in LlamaIndex?")
print(f"\nStep 5 ✅ Query Answer:\n   {ans}")
print("\n🎉 Level 1 pipeline complete!")

## Topic 15 — Index Persistence

In [ ]:
import os
from llama_index.core import StorageContext, load_index_from_storage

PERSIST_DIR = "./storage/level1_index"

if not os.path.exists(PERSIST_DIR):
    print("First run: building and saving index...")
    index = VectorStoreIndex(nodes)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"✅ Saved to {PERSIST_DIR}")
else:
    print("Loading index from disk (no embedding cost)...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    print(f"✅ Loaded from {PERSIST_DIR}")

qe = index.as_query_engine()
print("Index ready!")

---
## Level 1 — Summary & Assessment

### Architecture Recap
```
LlamaIndex Architecture
│
├── Data Layer
│   ├── Document  → Container for raw text + metadata
│   └── Node      → Atomic chunk; the retrieval unit
│
├── Index Layer
│   ├── VectorStoreIndex → Embeds + organizes Nodes
│   └── Settings         → Global LLM + embed config
│
├── Query Layer
│   ├── Retriever           → Finds relevant Nodes
│   ├── Response Synthesizer → LLM generates answer
│   ├── Query Engine        → Retriever + Synthesizer
│   └── Chat Engine         → Query Engine + Memory
│
└── Agent Layer
    ├── Tool      → Callable capability for agents
    ├── Agent     → LLM that dynamically uses tools
    └── Workflow  → Explicit event-driven pipeline
```

### Knowledge Checklist
- [ ] Can explain what LlamaIndex is and why it exists
- [ ] Understands the 3 pillars: Data, Indexing, Agents
- [ ] Knows difference between Document and Node
- [ ] Knows what an Index does and when to rebuild vs load
- [ ] Understands what Retriever returns and how
- [ ] Knows difference between Query Engine and Chat Engine
- [ ] Understands Response Synthesizer modes
- [ ] Knows what Tools are and why descriptions matter
- [ ] Understands Agent vs Workflow
- [ ] Can configure Settings globally
- [ ] Can persist and reload an index

---

## Exercise

Build a complete mini-RAG system from scratch:
1. Create 5+ Documents about a topic of your choice
2. Parse them into Nodes with `chunk_size=128`
3. Build a `VectorStoreIndex`
4. Use a Retriever — print top-3 nodes for a query
5. Use a Query Engine — ask 3 different questions
6. Persist the index; reload on second run
7. Verify reloaded index answers correctly

---

## Interview Questions

**Q1 (Beginner):** What problem does LlamaIndex solve?
> LLMs don't know your private data. LlamaIndex loads, indexes, and retrieves relevant pieces at query time to ground LLM responses.

**Q2 (Beginner):** What is the difference between a Document and a Node?
> Document = raw data container (entire file). Node = processed chunk, the actual unit stored in the index and retrieved. Documents are input; Nodes are what gets embedded and searched.

**Q3 (Beginner):** Why split documents into chunks?
> LLMs have token limits. Smaller, focused chunks are semantically coherent → better embeddings → more precise retrieval.

**Q4 (Intermediate):** Retriever vs Query Engine?
> Retriever returns Node objects (raw chunks). Query Engine returns a final answer string — internally it retrieves then calls LLM to synthesize.

**Q5 (Intermediate):** Why persist the index?
> Building requires embedding API calls (time + money). Persisting saves vectors to disk; loading skips embedding entirely.

**Q6 (Advanced):** When to use Workflow instead of Agent?
> Workflows in production — deterministic, reliable, debuggable. Agents for open-ended tasks where required steps aren't known in advance.

---

## Ready for Level 2?

- [ ] You understand all 10 core abstractions
- [ ] You can trace: raw text → Document → Node → Index → Retriever → Answer
- [ ] You can distinguish Query Engine vs Chat Engine vs Agent vs Workflow

**Say 'NEXT LEVEL' to continue to Level 2 — Data Loading** 🚀